# RescueSight: fine-tune the aerial person detector on Kaggle's free GPU

Trains YOLOv8n on VisDrone (people seen from drones), compares it with the stock model, and
leaves `person-visdrone.pt` (~6 MB) for the Jetson. Steps: `docs/training.md`.

**Before running (right-hand panel → Session options):**
1. **Accelerator: GPU T4** (one T4 is plenty; avoid P100, which recent PyTorch may not support).
2. **Internet: On** (Kaggle asks for phone verification the first time).

**Run it as a background job:** click **Save Version → Save & Run All (Commit)**. You can close
the browser; Kaggle runs it (up to ~12 h) and keeps the outputs. When it finishes, open the
version → **Output** and download `person-visdrone.pt` and `eval.md`.

Default: 50 epochs (early stopping after 15 without improvement). Rough guess, not measured:
a few hours on a T4. For a quicker run set `EPOCHS = 20` and `FRACTION = 0.5` below.


In [ ]:
# Settings
EPOCHS = 50          # early stopping usually ends sooner
FRACTION = 1.0       # share of the training tiles to use (0.5 = half, quicker)
BATCH = 32           # fits a 16 GB T4 easily for YOLOv8n at 640
ULTRALYTICS = "8.3.253"   # keep the same version on the Jetson
BRANCH = "claude"
SMOKE = False        # True = tiny CPU test of this notebook (not a real run)

import os
from pathlib import Path
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("RESCUE_NB_OUT", "nb_out")).resolve()
WORK = Path("/tmp/rescue") if ON_KAGGLE else Path(os.environ.get("RESCUE_NB_WORK", "nb_work")).resolve()
REPO = WORK / "HackUmass-2026"
DATA = WORK / "datasets"
DEVICE = "cpu" if SMOKE else "0"
if SMOKE:
    EPOCHS, FRACTION, BATCH = 1, 1.0, 8
OUT.mkdir(parents=True, exist_ok=True); DATA.mkdir(parents=True, exist_ok=True)
print(f"outputs -> {OUT}\nwork    -> {WORK}\nepochs {EPOCHS}, fraction {FRACTION}, batch {BATCH}, device {DEVICE}")


In [ ]:
import subprocess, sys

def run(*cmd, cwd=None):
    """Run a command, stream its output, stop the notebook if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)


In [ ]:
# 1. GPU check: uses Kaggle's preinstalled PyTorch (do not reinstall it)
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if DEVICE != "cpu":
    assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4' in Session options, then restart."
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 2**30:.0f} GB, compute capability {p.major}.{p.minor}")
TORCH_BEFORE = torch.__version__


In [ ]:
# 2. Ultralytics (must not replace Kaggle's CUDA PyTorch)
import importlib.metadata as md
try:
    have = md.version("ultralytics")
except md.PackageNotFoundError:
    have = None
if have != ULTRALYTICS:
    run(sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}")
torch_after = md.version("torch")
assert torch_after.split("+")[0] == TORCH_BEFORE.split("+")[0], (
    f"pip replaced torch {TORCH_BEFORE} with {torch_after}: restart the session and report this.")
print("ultralytics", md.version("ultralytics"), "| torch unchanged:", torch_after)


In [ ]:
# 3. Project code
if not (REPO / ".git").is_dir():
    run("git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/Abishhhekkk/HackUmass-2026.git", REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)


In [ ]:
# 4. VisDrone DET train + val (~1.6 GB, Ultralytics mirror; non-commercial research licence)
import urllib.request, zipfile
URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-{}.zip"
for split in (["val"] if SMOKE else ["train", "val"]):
    target = DATA / f"VisDrone2019-DET-{split}"
    if (target / "images").is_dir():
        print(split, "already present"); continue
    zpath = DATA / f"{split}.zip"
    print("downloading", split, "...", flush=True)
    urllib.request.urlretrieve(URL.format(split), zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA)
    zpath.unlink()
    print(split, len(list((target / "images").glob("*.jpg"))), "images")


In [ ]:
# 5. Tiles at native resolution + single-class 'person' labels
PERSON = DATA / ("visdrone-person-smoke" if SMOKE else "visdrone-person")
train_src = DATA / ("VisDrone2019-DET-val" if SMOKE else "VisDrone2019-DET-train")
if not (PERSON / "data.yaml").is_file():
    import shutil; shutil.rmtree(PERSON, ignore_errors=True)
    run(sys.executable, "scripts/prepare_visdrone.py", "--train", train_src,
        "--val", DATA / "VisDrone2019-DET-val", "--out", PERSON,
        *(["--limit", "20"] if SMOKE else []), cwd=REPO)
print(len(list((PERSON / "images" / "train").glob("*.jpg"))), "training tiles")


In [ ]:
# 6. Fine-tune YOLOv8n (COCO-pretrained) at the Jetson's 640 engine size
RUNS = OUT / "runs"
NAME = "smoke" if SMOKE else "person-yolov8n"
BEST = RUNS / NAME / "weights" / "best.pt"
if not BEST.is_file():
    import shutil; shutil.rmtree(RUNS / NAME, ignore_errors=True)
    run(sys.executable, "scripts/train_detector.py", "--data", PERSON / "data.yaml",
        "--epochs", EPOCHS, "--fraction", FRACTION, "--batch", BATCH, "--device", DEVICE,
        "--workers", min(4, os.cpu_count() or 1), "--project", RUNS, "--name", NAME, cwd=REPO)
print("best weights:", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# 7. Stock vs fine-tuned on the untouched VisDrone val images (full frames, tiled like the Jetson)
VAL = DATA / "VisDrone2019-DET-val"
lines = []
for label, weights in (("stock YOLOv8n (COCO)", REPO / "yolov8n.pt"), ("fine-tuned", BEST)):
    for tile in (640, 320):
        r = subprocess.run([sys.executable, "scripts/eval_detector.py", "--images", str(VAL / "images"),
                            "--labels", str(VAL / "annotations"), "--format", "visdrone",
                            "--weights", str(weights), "--imgsz", "640", "--conf", "0.1", "0.25",
                            "--tile", str(tile), "--device", DEVICE,
                            *(["--limit", "3"] if SMOKE else [])],
                           cwd=REPO, capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-2000:], r.stderr[-2000:]); raise SystemExit(f"evaluation failed: {label}, tile {tile}")
        table = [l for l in r.stdout.splitlines() if l.startswith("|")]
        lines += [f"## {label}, tile {tile}", *table, ""]
        print(lines[-len(table) - 2], *table, sep="\n")
(OUT / "eval.md").write_text("\n".join(lines))


In [ ]:
# 8. Files to download (Output tab): person-visdrone.pt + eval.md
import shutil
final = OUT / "person-visdrone.pt"
shutil.copy2(BEST, final)
print(f"{final} ({final.stat().st_size / 1e6:.1f} MB)")
print(open(OUT / "eval.md").read())
print("On the Jetson: python3 edge/export_model.py --model models/person-visdrone.pt "
      "--output models/person-visdrone.engine --imgsz 640")
